[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/DeepLearningKurs/blob/main/notebooks/Day_6_8/06_tag_6_8_pooling_methoden_padding_invarianz.ipynb)

# Tag 6-8 - 06 Pooling, Padding und Positionsinvarianz

Pooling bekommt Tensoren der Form `(N,C,H,W)` und reduziert meist `H,W`. Hier werden Inputs, Outputs, Padding, Stride und Positionsinvarianz explizit gezeigt.


In [ ]:
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = False

import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, Dataset, random_split

torch.manual_seed(RANDOM_STATE)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("CUDA verfügbar:", torch.cuda.is_available())
print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

from sklearn.datasets import load_digits


## `nn.MaxPool2d` und `nn.AvgPool2d`

Wichtige Parameter:

- `kernel_size`: Fenstergröße, zum Beispiel `2` für 2x2.
- `stride`: Schrittweite. Wenn `stride=2`, wird die räumliche Größe meist halbiert.
- `padding`: Randauffüllung. Dadurch können Randbereiche anders behandelt und Output-Shapes verändert werden.
- Input: `(batch, channels, height, width)`.
- Output: `(batch, channels, new_height, new_width)`.

Pooling hat keine trainierbaren Gewichte.


In [ ]:
digits = load_digits()
image = torch.tensor(digits.images[9], dtype=torch.float32).unsqueeze(0).unsqueeze(0) / 16.0
max_pool = nn.MaxPool2d(kernel_size=2, stride=2)
avg_pool = nn.AvgPool2d(kernel_size=2, stride=2)
min_pool = lambda x: -max_pool(-x)

outputs = {
    "Original": image,
    "MaxPool": max_pool(image),
    "AvgPool": avg_pool(image),
    "MinPool": min_pool(image),
}
fig, axes = plt.subplots(1, 4, figsize=(10, 2.8))
for ax, (name, tensor) in zip(axes, outputs.items()):
    ax.imshow(tensor.squeeze(), cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"{name}\n{tuple(tensor.shape[-2:])}")
    ax.axis("off")


## Padding-Funktion direkt kennenlernen

`torch.nn.functional.pad(input, pad, mode, value)` kann Tensoren manuell auffüllen. Für 2D-Bilder bedeutet `pad=(left, right, top, bottom)`.


In [ ]:
import torch.nn.functional as F

x = torch.arange(1, 17, dtype=torch.float32).reshape(1, 1, 4, 4)
padded = F.pad(x, pad=(1, 1, 1, 1), mode="constant", value=0)
print("Original shape:", x.shape)
print(x.squeeze())
print("Padded shape:", padded.shape)
print(padded.squeeze())


## Padding direkt im Pooling


In [ ]:
for padding in [0, 1]:
    pool = nn.MaxPool2d(kernel_size=2, stride=2, padding=padding)
    y = pool(x)
    print("padding=", padding, "output shape=", tuple(y.shape))
    print(y.squeeze())


## Padding in einem einfachen CNN

Padding wird in echten CNNs häufig direkt in der Convolution gesetzt. Bei einem 3x3-Filter sorgt `padding=1` dafür, dass aus einem 8x8-Bild wieder eine 8x8-Feature-Map entsteht. Erst das anschließende Pooling halbiert die Größe auf 4x4. So kann der Klassifikationskopf mit einer festen Eingabegröße arbeiten und der Bildrand geht nicht schon bei der ersten Convolution verloren.


In [ ]:
class EinfachesPaddingCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 4, kernel_size=3, padding=1),  # 8x8 -> 8x8
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),      # 8x8 -> 4x4
        )
        self.classifier = nn.Linear(4 * 4 * 4, 10)

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, start_dim=1)
        return self.classifier(x)


netz = EinfachesPaddingCNN().to(device)
bild = image.to(device)

with torch.no_grad():
    nach_conv = netz.features[0](bild)
    nach_pooling = netz.features(bild)
    logits = netz(bild)

print("Eingabe:      ", tuple(bild.shape))
print("Nach Conv2d:  ", tuple(nach_conv.shape), "-- padding=1 erhält 8x8")
print("Nach Pooling: ", tuple(nach_pooling.shape))
print("Klassen-Logits:", tuple(logits.shape))


## Positionsinvarianz


In [ ]:
img_a = torch.zeros(1, 1, 8, 8)
img_b = torch.zeros(1, 1, 8, 8)
img_a[:, :, 2, 2] = 1.0
img_b[:, :, 2, 3] = 1.0
pooled_a = max_pool(img_a)
pooled_b = max_pool(img_b)

fig, axes = plt.subplots(2, 2, figsize=(5, 5))
for ax, tensor, title in [
    (axes[0, 0], img_a, "Original A"),
    (axes[0, 1], img_b, "Original B verschoben"),
    (axes[1, 0], pooled_a, "Pool A"),
    (axes[1, 1], pooled_b, "Pool B"),
]:
    ax.imshow(tensor.squeeze(), cmap="gray", vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis("off")
print("Pool A == Pool B:", torch.equal(pooled_a, pooled_b))
